# EDA датасета Emergency Vehicle Siren Sounds

Разведочный анализ локального датасета EVSS из `data/raw/evss`.

Ожидаемая структура:

```text
data/raw/evss/
├── ambulance/
│   ├── sound_1.wav ... sound_200.wav
│   └── sound_1.png ... sound_200.png
├── firetruck/
│   ├── sound_201.wav ... sound_400.wav
│   └── sound_201.png ... sound_400.png
└── traffic/
    ├── sound_401.wav ... sound_600.wav
    └── sound_401.png ... sound_600.png
```

Цель ноутбука — понять, какие целевые классы проектной задачи реально покрываются этим набором, насколько он сбалансирован, какие есть технические неоднородности и какие риски нужно учесть при объединении EVSS с другими датасетами.

## Настройка

In [ ]:
from pathlib import Path

import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
from tqdm.notebook import tqdm

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 80)

RANDOM_STATE = 42
TARGET_CLASSES = [
    'siren_ambulance',
    'siren_police',
    'siren_firefighters',
    'car_horn',
    'car_idling',
    'car_braking',
    'motorcycle_acceleration',
    'truck_horn',
    'truck_idling',
    'tram_bell',
]

FOLDER_TO_TARGET = {
    'ambulance': 'siren_ambulance',
    'firetruck': 'siren_firefighters',
    'traffic': None,
}

FOLDER_TO_DESCRIPTION = {
    'ambulance': 'сирена скорой помощи',
    'firetruck': 'сирена пожарной машины',
    'traffic': 'общий дорожный/транспортный фон без точной разметки события',
}

PALETTE = {
    'ambulance': '#4C78A8',
    'firetruck': '#E45756',
    'traffic': '#54A24B',
}


In [ ]:
repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

EVSS_ROOT = repo_root / 'data' / 'raw' / 'evss'
assert EVSS_ROOT.exists(), f'Папка EVSS не найдена: {EVSS_ROOT}'

print(f'Корень проекта: {repo_root}')
print(f'Папка EVSS: {EVSS_ROOT}')
print('Подпапки:', ', '.join(sorted(path.name for path in EVSS_ROOT.iterdir() if path.is_dir())))


## Manifest

В EVSS нет отдельного CSV с метаданными, поэтому manifest строится напрямую из файловой структуры. Для каждого WAV фиксируются папка-класс, путь к парному PNG, длительность, частота дискретизации, число каналов, формат, размер файла и несколько лёгких аудио-признаков для EDA.

In [ ]:
def parse_sound_number(path: Path) -> int | None:
    stem = path.stem
    if stem.startswith('sound_') and stem.split('_')[-1].isdigit():
        return int(stem.split('_')[-1])
    return None


def audio_stats(path: Path) -> dict:
    info = sf.info(path)
    audio, sr = sf.read(path, always_2d=True)
    mono = audio.mean(axis=1).astype(float)
    rms = float(np.sqrt(np.mean(mono ** 2)))
    peak_abs = float(np.max(np.abs(mono)))
    mean_abs = float(np.mean(np.abs(mono)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(mono)[0]))
    spectral_centroid = float(np.mean(librosa.feature.spectral_centroid(y=mono, sr=sr)[0]))
    spectral_bandwidth = float(np.mean(librosa.feature.spectral_bandwidth(y=mono, sr=sr)[0]))
    return {
        'sample_rate': info.samplerate,
        'channels': info.channels,
        'frames': info.frames,
        'duration_sec': info.duration,
        'format': info.format,
        'subtype': info.subtype,
        'size_mb': path.stat().st_size / 1024 / 1024,
        'rms': rms,
        'peak_abs': peak_abs,
        'mean_abs': mean_abs,
        'zero_crossing_rate': zcr,
        'spectral_centroid_hz': spectral_centroid,
        'spectral_bandwidth_hz': spectral_bandwidth,
    }


rows = []
for wav_path in tqdm(sorted(EVSS_ROOT.rglob('*.wav')), desc='Чтение EVSS'):
    folder = wav_path.parent.name.lower()
    png_path = wav_path.with_suffix('.png')
    rows.append(
        {
            'filename': wav_path.name,
            'sound_id': parse_sound_number(wav_path),
            'folder': folder,
            'description': FOLDER_TO_DESCRIPTION.get(folder, 'неизвестная папка'),
            'target_class': FOLDER_TO_TARGET.get(folder),
            'filepath': wav_path,
            'png_path': png_path,
            'has_png': png_path.exists(),
            **audio_stats(wav_path),
        }
    )

manifest = (
    pd.DataFrame(rows)
    .sort_values(['folder', 'sound_id', 'filename'])
    .reset_index(drop=True)
)
manifest['duration_min'] = manifest['duration_sec'] / 60
manifest['duration_hour'] = manifest['duration_sec'] / 3600
manifest['is_direct_target'] = manifest['target_class'].notna()
manifest['near_clip_peak'] = manifest['peak_abs'] >= 0.99

print(f'WAV-файлов: {len(manifest):,}')
print(f'Суммарная длительность: {manifest["duration_sec"].sum():.1f} сек = {manifest["duration_hour"].sum():.3f} часа')
print(f'Файлов с парной PNG-спектрограммой: {manifest["has_png"].sum():,} из {len(manifest):,}')
display(manifest.drop(columns=['filepath', 'png_path']).head(10))


EVSS локально представлен очень аккуратно по количеству файлов: три папки по 200 WAV. Важно, что папка `traffic` не является точной меткой `car_horn`, `car_idling` или `car_braking`; это скорее общий дорожный фон, который может быть полезен как вспомогательный класс или негативный фон, но не как чистая целевая разметка этих событий.

## Покрытие целевых классов

In [ ]:
folder_summary = (
    manifest.groupby('folder')
    .agg(
        files=('filename', 'count'),
        target_class=('target_class', lambda s: next((x for x in s.dropna().unique()), None)),
        description=('description', 'first'),
        duration_sec=('duration_sec', 'sum'),
        sample_rates=('sample_rate', lambda s: sorted(s.unique())),
        channels=('channels', lambda s: sorted(s.unique())),
    )
    .reset_index()
    .sort_values('folder')
)
folder_summary['duration_min'] = folder_summary['duration_sec'] / 60

display(folder_summary)

coverage_rows = []
for target_class in TARGET_CLASSES:
    exact_files = int((manifest['target_class'] == target_class).sum())
    if target_class in {'car_horn', 'car_idling', 'car_braking'}:
        note = 'В EVSS есть только общая папка traffic; точной разметки этого события нет.'
        candidate_files = int((manifest['folder'] == 'traffic').sum())
    else:
        note = 'Прямое соответствие папке EVSS.' if exact_files else 'В EVSS не найдено прямого соответствия.'
        candidate_files = 0
    coverage_rows.append(
        {
            'target_class': target_class,
            'direct_files': exact_files,
            'candidate_auxiliary_files': candidate_files,
            'coverage_status': 'direct' if exact_files else ('candidate_only' if candidate_files else 'missing'),
            'comment': note,
        }
    )

coverage = pd.DataFrame(coverage_rows)
display(coverage)


In [ ]:
coverage_plot = coverage.copy()
coverage_plot['status_ru'] = coverage_plot['coverage_status'].map(
    {
        'direct': 'прямое покрытие',
        'candidate_only': 'только кандидат/фон',
        'missing': 'нет покрытия',
    }
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.countplot(
    data=coverage_plot,
    x='status_ru',
    order=['прямое покрытие', 'только кандидат/фон', 'нет покрытия'],
    palette=['#4C78A8', '#F2CF5B', '#BAB0AC'],
    ax=axes[0],
)
axes[0].set_title('Статус покрытия целевых классов')
axes[0].set_xlabel('Статус')
axes[0].set_ylabel('Классы')
axes[0].bar_label(axes[0].containers[0])

plot_rows = coverage_plot.query('direct_files > 0 or candidate_auxiliary_files > 0').melt(
    id_vars=['target_class'],
    value_vars=['direct_files', 'candidate_auxiliary_files'],
    var_name='coverage_type',
    value_name='files',
)
sns.barplot(data=plot_rows, x='files', y='target_class', hue='coverage_type', ax=axes[1])
axes[1].set_title('Количество файлов по покрытым/кандидатным классам')
axes[1].set_xlabel('Файлы')
axes[1].set_ylabel('Целевой класс')
axes[1].legend(title='Тип покрытия')

plt.tight_layout()
plt.show()


Прямое покрытие есть только у `siren_ambulance` и `siren_firefighters`: по 200 файлов на каждый класс. Для `car_horn`, `car_idling` и `car_braking` папка `traffic` может дать дорожный контекст, но без дополнительной ручной проверки её нельзя считать корректной supervised-разметкой. Остальные целевые классы EVSS не покрывает.

## Баланс и длительность

In [ ]:
balance = (
    manifest.groupby('folder')
    .agg(
        files=('filename', 'count'),
        duration_sec_total=('duration_sec', 'sum'),
        duration_sec_mean=('duration_sec', 'mean'),
        duration_sec_median=('duration_sec', 'median'),
        duration_sec_min=('duration_sec', 'min'),
        duration_sec_max=('duration_sec', 'max'),
        duration_sec_std=('duration_sec', 'std'),
    )
    .reset_index()
)
balance['duration_min_total'] = balance['duration_sec_total'] / 60

display(balance)

total_files = len(manifest)
total_duration_sec = manifest['duration_sec'].sum()
print(f'Всего файлов: {total_files:,}')
print(f'Всего аудио: {total_duration_sec:.1f} сек = {total_duration_sec / 60:.1f} мин = {total_duration_sec / 3600:.3f} часа')
print(f'Средняя длительность клипа: {manifest["duration_sec"].mean():.3f} сек')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(data=balance, x='folder', y='files', hue='folder', palette=PALETTE, legend=False, ax=axes[0])
axes[0].set_title('Количество WAV по папкам')
axes[0].set_xlabel('Папка')
axes[0].set_ylabel('Файлы')
axes[0].bar_label(axes[0].containers[0])

sns.barplot(data=balance, x='folder', y='duration_min_total', hue='folder', palette=PALETTE, legend=False, ax=axes[1])
axes[1].set_title('Суммарная длительность')
axes[1].set_xlabel('Папка')
axes[1].set_ylabel('Минуты')
axes[1].bar_label(axes[1].containers[0], fmt='%.1f')

sns.boxplot(data=manifest, x='duration_sec', y='folder', hue='folder', palette=PALETTE, legend=False, ax=axes[2])
axes[2].set_title('Длительность клипов')
axes[2].set_xlabel('Секунды')
axes[2].set_ylabel('Папка')

plt.tight_layout()
plt.show()


По количеству файлов датасет идеально сбалансирован: 200 клипов на каждую папку. Длительности также почти стандартизированы вокруг 3 секунд, поэтому при обучении не ожидается сильного перекоса из-за длины клипа. Ограничение другое: суммарно здесь около получаса аудио, а прямых целевых классов только два.

## Технические параметры

In [ ]:
technical_summary = (
    manifest.groupby('folder')
    .agg(
        files=('filename', 'count'),
        sample_rates=('sample_rate', lambda s: sorted(s.unique())),
        channels=('channels', lambda s: sorted(s.unique())),
        formats=('format', lambda s: sorted(s.unique())),
        subtypes=('subtype', lambda s: sorted(s.unique())),
        frames_min=('frames', 'min'),
        frames_max=('frames', 'max'),
        size_mb_mean=('size_mb', 'mean'),
        size_mb_total=('size_mb', 'sum'),
    )
    .reset_index()
)

display(technical_summary)

display(
    manifest[['folder', 'sample_rate', 'channels', 'format', 'subtype']]
    .value_counts()
    .reset_index(name='files')
    .sort_values(['folder', 'sample_rate', 'channels'])
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sr_counts = manifest.value_counts(['folder', 'sample_rate']).reset_index(name='files')
sns.barplot(data=sr_counts, x='sample_rate', y='files', hue='folder', palette=PALETTE, ax=axes[0])
axes[0].set_title('Частоты дискретизации по папкам')
axes[0].set_xlabel('Sample rate, Hz')
axes[0].set_ylabel('Файлы')
axes[0].legend(title='Папка')

channel_counts = manifest.value_counts(['folder', 'channels']).reset_index(name='files')
sns.barplot(data=channel_counts, x='channels', y='files', hue='folder', palette=PALETTE, ax=axes[1])
axes[1].set_title('Количество каналов')
axes[1].set_xlabel('Каналы')
axes[1].set_ylabel('Файлы')
axes[1].legend(title='Папка')

plt.tight_layout()
plt.show()


Все файлы — WAV PCM 16-bit, но частота дискретизации неоднородна: встречаются 44.1 кГц, 48 кГц и отдельные 22.05 кГц. Число каналов тоже не полностью одинаковое: `firetruck` и `traffic` в этой копии стерео, а в `ambulance` есть и моно, и стерео. Для моделирования нужен единый preprocessing: приведение к mono и общей частоте дискретизации.

## Уровни сигнала

In [ ]:
level_summary = (
    manifest.groupby('folder')
    .agg(
        rms_mean=('rms', 'mean'),
        rms_median=('rms', 'median'),
        rms_min=('rms', 'min'),
        rms_max=('rms', 'max'),
        peak_median=('peak_abs', 'median'),
        peak_max=('peak_abs', 'max'),
        near_clip_files=('near_clip_peak', 'sum'),
        quietest_file=('rms', lambda s: manifest.loc[s.idxmin(), 'filename']),
        loudest_file=('rms', lambda s: manifest.loc[s.idxmax(), 'filename']),
    )
    .reset_index()
)

display(level_summary)

display(
    manifest.sort_values('rms')
    [['folder', 'filename', 'rms', 'peak_abs', 'duration_sec', 'sample_rate', 'channels']]
    .head(10)
)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=manifest, x='rms', y='folder', hue='folder', palette=PALETTE, legend=False, ax=axes[0])
axes[0].set_title('RMS по папкам')
axes[0].set_xlabel('RMS')
axes[0].set_ylabel('Папка')

sns.boxplot(data=manifest, x='peak_abs', y='folder', hue='folder', palette=PALETTE, legend=False, ax=axes[1])
axes[1].axvline(0.99, color='#B279A2', linestyle='--', linewidth=1.5, label='0.99')
axes[1].set_title('Пиковая амплитуда')
axes[1].set_xlabel('Peak abs')
axes[1].set_ylabel('Папка')
axes[1].legend()

clip_counts = manifest.groupby('folder')['near_clip_peak'].sum().reset_index(name='files')
sns.barplot(data=clip_counts, x='folder', y='files', hue='folder', palette=PALETTE, legend=False, ax=axes[2])
axes[2].set_title('Файлы с peak >= 0.99')
axes[2].set_xlabel('Папка')
axes[2].set_ylabel('Файлы')
axes[2].bar_label(axes[2].containers[0])

plt.tight_layout()
plt.show()


Уровни заметно различаются между папками: `traffic` тише, а у `firetruck` много файлов с пиком около 1.0. Это не делает датасет непригодным, но при обучении лучше применять нормализацию громкости и отдельно проверить клипы с `peak_abs >= 0.99`, чтобы не смешивать признак класса с артефактом уровня записи.

## Спектральные признаки

In [ ]:
spectral_summary = (
    manifest.groupby('folder')
    .agg(
        zcr_mean=('zero_crossing_rate', 'mean'),
        zcr_median=('zero_crossing_rate', 'median'),
        centroid_mean_hz=('spectral_centroid_hz', 'mean'),
        centroid_median_hz=('spectral_centroid_hz', 'median'),
        bandwidth_mean_hz=('spectral_bandwidth_hz', 'mean'),
        bandwidth_median_hz=('spectral_bandwidth_hz', 'median'),
    )
    .reset_index()
)

display(spectral_summary)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=manifest, x='zero_crossing_rate', y='folder', hue='folder', palette=PALETTE, legend=False, ax=axes[0])
axes[0].set_title('Zero crossing rate')
axes[0].set_xlabel('ZCR')
axes[0].set_ylabel('Папка')

sns.boxplot(data=manifest, x='spectral_centroid_hz', y='folder', hue='folder', palette=PALETTE, legend=False, ax=axes[1])
axes[1].set_title('Спектральный центроид')
axes[1].set_xlabel('Hz')
axes[1].set_ylabel('Папка')

sns.scatterplot(
    data=manifest,
    x='spectral_centroid_hz',
    y='rms',
    hue='folder',
    palette=PALETTE,
    alpha=0.75,
    ax=axes[2],
)
axes[2].set_title('RMS vs спектральный центроид')
axes[2].set_xlabel('Спектральный центроид, Hz')
axes[2].set_ylabel('RMS')
axes[2].legend(title='Папка')

plt.tight_layout()
plt.show()


Простые спектральные признаки уже разделяют папки: `firetruck` в среднем ярче по спектральному центроиду и ZCR, `traffic` ниже по энергии и частотному центру, `ambulance` занимает промежуточную область. Это полезный sanity-check, но для модели лучше использовать полные time-frequency признаки, потому что сирены различаются не только средним спектром, но и периодической модуляцией во времени.

## Парные PNG-спектрограммы

In [ ]:
asset_summary = []
for folder_path in sorted(path for path in EVSS_ROOT.iterdir() if path.is_dir()):
    wav_count = len(list(folder_path.glob('*.wav')))
    png_count = len(list(folder_path.glob('*.png')))
    py_count = len(list(folder_path.glob('*.py')))
    asset_summary.append(
        {
            'folder': folder_path.name,
            'wav_files': wav_count,
            'png_files': png_count,
            'python_files': py_count,
            'wav_with_png': int(manifest.query('folder == @folder_path.name')['has_png'].sum()),
        }
    )

asset_summary = pd.DataFrame(asset_summary)
display(asset_summary)


В папках есть PNG-спектрограммы для каждого WAV. Их можно использовать для ручного просмотра качества, но для обучения предпочтительнее строить спектрограммы из WAV в едином preprocessing pipeline: так контролируются sample rate, mono/stereo, нормализация, параметры STFT и mel-шкалы.

## Волновые формы и mel-спектрограммы

In [ ]:
def representative_rows(df: pd.DataFrame) -> pd.DataFrame:
    reps = []
    for folder, group in df.groupby('folder'):
        median_rms = group['rms'].median()
        row = group.iloc[(group['rms'] - median_rms).abs().argsort().iloc[0]]
        reps.append(row)
    return pd.DataFrame(reps).sort_values('folder').reset_index(drop=True)

examples = representative_rows(manifest)
display(examples[['folder', 'filename', 'target_class', 'duration_sec', 'sample_rate', 'channels', 'rms', 'peak_abs']])

fig, axes = plt.subplots(len(examples), 2, figsize=(15, 3.4 * len(examples)))
if len(examples) == 1:
    axes = np.array([axes])

for row_idx, row in enumerate(examples.itertuples(index=False)):
    signal, sr = librosa.load(row.filepath, sr=None, mono=True)
    librosa.display.waveshow(signal, sr=sr, ax=axes[row_idx, 0], color=PALETTE.get(row.folder, '#4C78A8'))
    axes[row_idx, 0].set_title(f'Волновая форма: {row.folder} / {row.filename}')
    axes[row_idx, 0].set_xlabel('Время, сек')
    axes[row_idx, 0].set_ylabel('Амплитуда')

    mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=96)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', cmap='magma', ax=axes[row_idx, 1])
    axes[row_idx, 1].set_title(f'Mel-спектрограмма: {row.folder} / {row.filename}')
    axes[row_idx, 1].set_xlabel('Время, сек')
    axes[row_idx, 1].set_ylabel('Mel')

plt.tight_layout()
plt.show()


На примерах видно, что все клипы короткие и стандартизированы по длине, но отличаются по тембру и уровню. Для `ambulance` и `firetruck` ожидаемо важны временные паттерны сирены, а `traffic` выглядит как более спокойный фон; это подтверждает, что `traffic` нельзя автоматически разложить на `car_horn`, `car_idling` и `car_braking` без дополнительной разметки.

## Прослушивание примера

In [ ]:
sample = examples.query("folder == 'ambulance'").iloc[0]
print(sample[['folder', 'filename', 'target_class', 'duration_sec', 'sample_rate', 'channels']])
ipd.Audio(sample['filepath'])


## Итоги

- В локальной копии EVSS найдено 600 WAV-файлов: `ambulance`, `firetruck` и `traffic` по 200 файлов в каждой папке.
- Датасет сбалансирован по папкам и почти стандартизирован по длительности: клипы около 3 секунд, суммарно примерно 30 минут аудио.
- Прямо покрываются только два целевых класса проекта: `siren_ambulance` и `siren_firefighters`.
- `traffic` лучше считать вспомогательным дорожным фоном или кандидатным источником для ручной доразметки, но не готовой разметкой `car_horn`, `car_idling` или `car_braking`.
- В EVSS нет прямого покрытия для `siren_police`, `motorcycle_acceleration`, `truck_horn`, `truck_idling` и `tram_bell`.
- Перед объединением с другими датасетами нужно привести WAV к единому формату: mono, общий sample rate, единая нормализация уровня и одинаковая схема построения признаков.
- Отдельный риск — неодинаковые уровни громкости и файлы с пиками около 1.0, особенно в `firetruck`; такие клипы стоит проверить на клиппинг или обработать нормализацией.

Практический вывод: EVSS полезен как компактный источник сирен скорой и пожарной машины, но не закрывает весь целевой набор классов. Для полной задачи его нужно объединять с UrbanSound8K, ESC-50, локальными записями или другим источником с точной разметкой недостающих транспортных событий.